# 酒店评论问答系统

In [1]:
import os
from pathlib import Path
from lib import HotelReviewRAG, print_rag_result

# 项目路径配置
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
print(f"离线数据目录: {DATA_DIR}")

# 检查环境变量
required_env = {
    "DASHSCOPE_API_KEY": os.getenv("DASHSCOPE_API_KEY"),
    "DASHVECTOR_API_KEY": os.getenv("DASHVECTOR_API_KEY"),
    "DASHVECTOR_HOTEL_ENDPOINT": os.getenv("DASHVECTOR_HOTEL_ENDPOINT"),
}

missing = [k for k, v in required_env.items() if not v]
if missing:
    raise EnvironmentError(f"缺少环境变量: {', '.join(missing)}")

print("环境变量检测成功:")
for key, value in required_env.items():
    print(f"- {key}: {value[:10]}...{value[-4:]}")

d:\anaconda3\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


离线数据目录: d:\dyx\DSBA\课程\1-2\大模型技术原理与商业应用\Exp3\data
环境变量检测成功:
- DASHSCOPE_API_KEY: sk-2824fcf...ca0c
- DASHVECTOR_API_KEY: sk-BSq8DoP...6A67
- DASHVECTOR_HOTEL_ENDPOINT: vrs-cn-aaf....com


In [2]:
# 初始化 RAG 系统
rag_system = HotelReviewRAG(
    api_key=required_env["DASHSCOPE_API_KEY"],
    dashvector_api_key=required_env["DASHVECTOR_API_KEY"],
    dashvector_endpoint=required_env["DASHVECTOR_HOTEL_ENDPOINT"],
    data_dir=DATA_DIR
)

print("✅ RAG系统初始化成功")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\26524\AppData\Local\Temp\jieba.cache
Loading model cost 0.612 seconds.
Prefix dict has been built successfully.


✅ RAG系统初始化成功


## 1. 意图检测与意图扩展演示

In [3]:
# 测试意图检测和意图扩展
test_queries = [
    "酒店的床舒服吗？睡眠质量怎么样？",
    "前台服务态度怎么样？",
    "早餐有广式点心吗？",
    "泳池水温怎么样？",
    "地铁怎么走？",
    "大床房隔音好吗？",
    "最近房间卫生怎么样？"
]

print("意图检测与意图扩展演示\n" + "="*60)
for query in test_queries:
    intent, confidence = rag_system.detect_intent(query)
    # 意图扩展
    expand_queries = rag_system.expand_intent(query)
    print(f"查询: {query}")
    print(f"意图检测: {intent}")
    print(f"置信度: {confidence:.2%}")
    if expand_queries:
        print("意图扩展:")
        for exp_q, weight in expand_queries:
            print(f"  • {exp_q} (weight={weight})")
    print()

意图检测与意图扩展演示
查询: 酒店的床舒服吗？睡眠质量怎么样？
意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None}
置信度: 100.00%
意图扩展:
  • 床的尺寸和材质如何？ (weight=0.4)
  • 房间隔音效果好吗？ (weight=0.3)
  • 是否有足够的睡眠环境（如光线、噪音等）？ (weight=0.3)

查询: 前台服务态度怎么样？
意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None}
置信度: 100.00%
意图扩展:
  • 前台人员是否及时响应客人需求？ (weight=0.6)
  • 前台办理入住和退房的效率如何？ (weight=0.3)
  • 酒店是否有24小时前台服务？ (weight=0.1)

查询: 早餐有广式点心吗？
意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None}
置信度: 100.00%
意图扩展:
  • 酒店是否提供广式点心？ (weight=0.6)
  • 早餐的种类是否丰富？ (weight=0.3)
  • 是否有其他地方特色食物？ (weight=0.1)

查询: 泳池水温怎么样？
意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None}
置信度: 100.00%
意图扩展:
  • 泳池是否全天开放？ (weight=0.4)
  • 泳池水质是否清澈？ (weight=0.3)
  • 是否有室内泳池或恒温泳池？ (weight=0.3)

查询: 地铁怎么走？
意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None}
置信度: 100.00%
意图扩展:
  • 地铁站距离酒店有多远？ (weight=0.4)
  • 酒店附近有地铁站吗？ (weight=0.3)
  • 地铁运营时间

## 2. 端到端问答演示

In [4]:
# 测试用例：床铺舒适度和睡眠质量
test_query = "酒店的床舒服吗？睡眠质量怎么样？"
result = rag_system.query(test_query, enable_hyde=True)
print_rag_result(result)

📝 查询: 酒店的床舒服吗？睡眠质量怎么样？

💬 最终答案:
--------------------------------------------------------------------------------
### 1. 正面评价：床品舒适，睡眠质量良好

根据多位住客的反馈，酒店的床品整体评价非常积极。多位客人提到“床褥松软温暖”、“床很舒服”、“睡眠很好”，尤其是花园双床房和花园大床房的住客对床垫的舒适度表示满意。部分客人还特别提到枕头有多种选择，增加了睡眠的舒适性。

此外，酒店的空调系统在干湿度控制方面表现良好，避免了传统酒店中常见的干燥问题，有助于提升整体睡眠体验。部分客人还提到房间隔音效果不错，有助于营造安静的休息环境。

### 2. 需要注意的方面：房间空间与卫生间的布局

虽然多数客人对床品和睡眠质量感到满意，但也有部分客人指出了一些需要注意的地方。例如，有客人提到部分房间面积较小，导致开放式卫生间的设计，使得马桶间和淋浴间空间较为局促，可能会影响使用体验。此外，部分房间的墙纸因年久而显得有些陈旧，但这并不影响整体居住体验。

另外，有客人提到早餐虽然种类丰富，但未见明显的地方特色，如广州特色的叉烧包等，若对地方风味有较高期待，可能需要额外准备或前往周边餐饮地点。

### 3. 小贴士：推荐楼层与早餐选择

- **早餐推荐**：建议选择30楼的旋转餐厅用餐，视野较好且每桌之间有间隔，私密性较强。
- **房间选择**：如果对空间有较高要求，可优先考虑花园双床房或大床房，这类房间在空间布局上相对更合理。
- **服务体验**：前台员工如Betty表现出色，入住体验专业热情，值得称赞。

总体而言，该酒店的床品和睡眠质量受到大多数住客的认可，是其一大优势，同时在细节上也存在一些可以优化的地方，建议根据个人需求进行选择。
--------------------------------------------------------------------------------

🎯 意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None} (置信度: 100.00%)

🔍 意图扩展:
  • 床的尺寸是否合适？ (weight=0.

In [5]:
# 测试用例：早餐
test_query = "早餐怎么样？"
result = rag_system.query(test_query, enable_hyde=True)
print_rag_result(result)

📝 查询: 早餐怎么样？

💬 最终答案:
--------------------------------------------------------------------------------
### 1. 正面评价（大多数住客的满意点）

多数住客对酒店的早餐给予了积极评价，普遍认为早餐**品种丰富、服务周到**。例如：

- **评论2**提到“早餐丰盛，而且国际化，又有本地特色”，表明早餐不仅种类多样，还兼顾了地方风味。
- **评论3**表示“早餐很不错”，并具体描述了在不同餐厅（瀑布餐厅和桃园厅）的早餐体验，说明早餐供应丰富且有变化。
- **评论5**指出“早餐品类丰富”，并特别表扬了服务员Lili的态度“非常暖心”，体现出良好的服务体验。
- **评论4**也提到“1楼的早餐服务就很棒，服务员也很有礼貌”，说明在部分楼层或区域的早餐服务体验非常出色。

总体来看，大部分住客对早餐的**质量、种类和服务态度**表示满意，尤其在**非30楼的早餐区域**，体验较为理想。

---

### 2. 需要注意的方面（可能的不足）

尽管多数住客对早餐表示满意，但也有一些负面反馈需要注意：

- **评论4**提到“30楼的早餐服务不如1楼的早餐服务，服务员说话也不好听”，这可能影响部分住客的早餐体验。该住客表示只在30楼吃了一次早餐，之后选择在1楼用餐。
- 另外，也有住客提到“早餐真心一般，品种少”（评论1），虽然该评论评分未明确，但反映出个别住客对早餐的不满。
- 有住客提到“带娃床有点小”（评论3），虽然这不是早餐相关的问题，但间接说明酒店在设计上可能对家庭住客不够友好，也可能影响整体用餐体验。

因此，如果住客对早餐体验有较高期望，建议优先选择**1楼或其他非30楼的早餐区域**，以获得更佳的服务体验。

---

### 3. 小贴士

- **推荐早起用餐**：部分住客反映，早餐时段人流较多，建议尽早前往以避免排队。
- **选择合适早餐区域**：根据评论，**1楼的早餐服务更为优质**，而**30楼的早餐服务体验较差**，建议优先选择1楼或其它非30楼的早餐地点。
- **提前确认早餐安排**
-----------------------------------------------------------------------

In [6]:
# 测试用例：地理位置和交通
test_query = "地理位置怎么样？交通方便吗？"
result = rag_system.query(test_query, enable_hyde=True)
print_rag_result(result)

📝 查询: 地理位置怎么样？交通方便吗？

💬 最终答案:
--------------------------------------------------------------------------------
### 1. 正面评价：地理位置优越，交通便利

多数住客对酒店的地理位置给予了高度评价。多位客人提到，酒店位于地铁站附近，出行非常方便。例如，评论1和评论5都指出酒店靠近“淘金站”地铁站，属于广州地铁5号线，便于前往城市各个区域。此外，评论2和评论4也提到“地理位置优越”、“交通方便”，说明该酒店在交通便利性方面是其主要优势之一。

部分客人还提到，酒店周边生活配套完善，适合游客或商务出行，能够快速到达景点、商圈或交通枢纽。

### 2. 需要注意的方面：房间陈旧、设施老化

尽管地理位置受到好评，但也有部分客人提到房间存在陈旧问题。例如，评论1提到“房间明显陈旧，希望有机会可以翻新”，评论3则指出“沙发有些脱皮了”。这些反馈表明，虽然酒店整体服务和位置不错，但部分客房内部设施可能需要更新。

此外，评论2提到“房间面积不大”，这可能对追求宽敞空间的客人来说是一个需要注意的点。

### 3. 小贴士：推荐入住及周边体验

- **推荐入住房型**：若注重性价比和交通便利，可考虑“花园大床房”或“城央绿意双床房”，部分客人表示这类房型舒适度较高。
- **早餐体验**：顶层的360度旋转餐厅提供早间餐食，如烟熏三文鱼等，适合喜欢精致早餐的客人。
- **周边推荐**：酒店附近有“荔湾庭”早午茶，环境佳、味道好，且性价比高，建议尝试。
- **行李协助**：酒店工作人员态度友好，尤其对带老人和孩子的家庭，有主动帮助搬运行李和提供路线指引的服务。

综上所述，该酒店在地理位置和交通便利性方面表现突出，是广州出行的优选之一，但部分客房设施可能略显陈旧，建议根据个人需求选择房型。
--------------------------------------------------------------------------------

🎯 意图检测: {'room_type': None, 'fuzzy_room_type': None, 'time_sensitivity': None} (置信度: 100.00%)

🔍 意图扩展:
  • 